Tools:
Allows frontier model the ability to connect to external functions


common use cases for tools:
fetch data or add knowledge or context
take action like booking a meeting 
perform calculations or run code
modify the ui

agentic ai:
2 ideas , core ideas behind agentic ai

a tool can be used to make another call to an llm
a tool can be used to track todo list and track progress towards a goal



In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

c:\Users\jagad\Desktop\llm_learning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Initialization

load_dotenv(override=True)
gemini_api_key = os.getenv('GOOGLE_API_KEY')
if gemini_api_key:
    print(f"Gemini API Key exists and begins {gemini_api_key[:8]}")
else:
    print("Gemini API Key not set")
    
MODEL = "gemini-3.5-flash-lite"
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/",api_key=gemini_api_key)


# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


Gemini API Key exists and begins AQ.Ab8RN


In [9]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [10]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


In [11]:

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [12]:
get_ticket_price("london")

Tool called for city london


'The price of a ticket to london is $799'

In [13]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [14]:
tools = [{"type": "function", "function": price_function}]

In [15]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [37]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    # print(response)
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        # print(message)
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        print(messages)

        response = gemini.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [32]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    print(tool_call)

    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [38]:
gr.ChatInterface(fn=chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7882
* To create a public link, set `share=True` in `launch()`.


ChatCompletionMessageFunctionToolCall(id='call_732111', function=Function(arguments='{"destination_city":"Tokyo"}', name='get_ticket_price'), type='function', extra_content={'google': {'thought_signature': 'El4KXAFpFH0TcjRn+sC2lA7gFKAm58mhOlQCUHfBdTcLIUjmaJ9Argwsa9lWIBO/agbEq5x775ujOHNcc+0o7bm4mG7KVFQnn9V0tcJVnJBLEbfscF3p6i7VB0VUlOYE'}})
Tool called for city Tokyo
[{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}, {'role': 'user', 'content': [{'text': 'Hello', 'type': 'text'}]}, {'role': 'assistant', 'content': [{'text': 'Hello! Welcome to FlightAI, how can I help you today?', 'type': 'text'}]}, {'role': 'user', 'content': 'Tokyo'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_732111', function=

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    # print(response)
    if response.choices[0].finish_reason=="tool_calls":
        
        message = response.choices[0].message
        # print(message)
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=MODEL, messages=messages)
        
    return response.choices[0].message.content

In [47]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    print(responses)
    return responses

In [ ]:
gr.ChatInterface(fn=chat).launch(inbrowser=True)

In [54]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [55]:
import sqlite3


In [56]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [58]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [59]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'No price data available for this city'

In [60]:
def set_ticket_price(city,price):
    with sqlite3.connect(DB) as conn:
        cursor=conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [61]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [62]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [ ]:
gr.ChatInterface(fn=chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7889
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for London
[{'role': 'tool', 'content': 'Ticket price to London is $799.0', 'tool_call_id': 'call_644650'}]
DATABASE TOOL CALLED: Getting price for London
DATABASE TOOL CALLED: Getting price for Tokyo
[{'role': 'tool', 'content': 'Ticket price to London is $799.0', 'tool_call_id': 'call_672673'}, {'role': 'tool', 'content': 'Ticket price to Tokyo is $1420.0', 'tool_call_id': 'call_672674'}]
